# DataFrames

DataFrames are a fundamental object type for representing data sets. A DataFrame can be viewed in two ways:
* a sequence of named columns that each describe a single aspect of all entries in a data set, or
* a sequence of rows that each contain all information about a single entry in a data set.

In order to use DataFrames, import the module called `polars`.

In [1]:
import polars as pl
path_data = '../../assets/data/'

Empty DataFrames can be created using `pl.DataFrame`. An empty DataFrame is useful because it can be extended to contain new rows and columns.

In [2]:
pl.DataFrame()

shape: (0, 0)
┌┐
╞╡
└┘

A DataFrame is built from named columns. Each column is a Polars Series. To create a DataFrame with one column, pass a Series to `pl.DataFrame`.

Below, we begin each example with a DataFrame that has one column.

In [3]:
pl.DataFrame([
    pl.Series("Number of petals", [8, 34, 5])
])

Number of petals
i64
8
34
5


To add two (or more) new columns, pass a Polars Series for each column. All columns must have the same length, or an error will occur.

In [4]:
pl.DataFrame([
    pl.Series("Number of petals", [8, 34, 5]),
    pl.Series("Name", ["lotus", "sunflower", "rose"]),
])

Number of petals,Name
i64,str
8,"""lotus"""
34,"""sunflower"""
5,"""rose"""


We can give this DataFrame a name, and then extend the DataFrame with another column.

In [5]:
flowers = pl.DataFrame([
    pl.Series("Number of petals", [8, 34, 5]),
    pl.Series("Name", ["lotus", "sunflower", "rose"]),
])

flowers.with_columns(
    pl.Series("Color", ["pink", "yellow", "red"])
)

Number of petals,Name,Color
i64,str,str
8,"""lotus""","""pink"""
34,"""sunflower""","""yellow"""
5,"""rose""","""red"""


The `with_columns` method creates a new DataFrame each time it is called, so the original DataFrame is not affected. For example, the DataFrame `flowers` still has only the two columns that it had when it was created.

In [6]:
flowers

Number of petals,Name
i64,str
8,"""lotus"""
34,"""sunflower"""
5,"""rose"""


Creating DataFrames in this way involves a lot of typing. If the data have already been entered somewhere, it is usually possible to use Python to read it into a DataFrame, instead of typing it all in cell by cell.

Often, DataFrames are created from files that contain comma-separated values. Such files are called CSV files.

Below, we use `pl.read_csv` to read a CSV file that contains some of the data used by Minard in his graphic about Napoleon's Russian campaign. The data are placed in a DataFrame named `minard`.

POLARS_CHANGE: This file separates rows with carriage returns, so `read_csv` is given `eol_char="\r"`.

In [7]:
minard = pl.read_csv(path_data + 'minard.csv', eol_char='\r')
minard

Longitude,Latitude,City,Direction,Survivors
f64,f64,str,str,i64
32.0,54.8,"""Smolensk""","""Advance""",145000
33.2,54.9,"""Dorogobouge""","""Advance""",140000
34.4,55.5,"""Chjat""","""Advance""",127100
37.6,55.8,"""Moscou""","""Advance""",100000
34.3,55.2,"""Wixma""","""Retreat""",55000
32.0,54.6,"""Smolensk""","""Retreat""",24000
30.4,54.4,"""Orscha""","""Retreat""",20000
26.8,54.3,"""Moiodexno""","""Retreat""",12000


We will use this small DataFrame to demonstrate some useful DataFrame methods. We will then use those same methods, and develop other methods, on much larger DataFrames of data.

**The Size of the DataFrame**

The attribute `width` gives the number of columns in the DataFrame, and `height` the number of rows.

In [8]:
minard.width

5

In [9]:
minard.height

8

**Column Labels**

The attribute `columns` lists the names of all the columns. With `minard` we don't gain much by this, but it can be very useful for DataFrames that are so large that not all columns are visible on the screen.

In [10]:
minard.columns

['Longitude', 'Latitude', 'City', 'Direction', 'Survivors']

We can change column labels using the `rename` method. This creates a new DataFrame and leaves `minard` unchanged.

In [11]:
minard.rename({"City": "City Name"})

Longitude,Latitude,City Name,Direction,Survivors
f64,f64,str,str,i64
32.0,54.8,"""Smolensk""","""Advance""",145000
33.2,54.9,"""Dorogobouge""","""Advance""",140000
34.4,55.5,"""Chjat""","""Advance""",127100
37.6,55.8,"""Moscou""","""Advance""",100000
34.3,55.2,"""Wixma""","""Retreat""",55000
32.0,54.6,"""Smolensk""","""Retreat""",24000
30.4,54.4,"""Orscha""","""Retreat""",20000
26.8,54.3,"""Moiodexno""","""Retreat""",12000


However, this method does not change the original DataFrame.

In [12]:
minard

Longitude,Latitude,City,Direction,Survivors
f64,f64,str,str,i64
32.0,54.8,"""Smolensk""","""Advance""",145000
33.2,54.9,"""Dorogobouge""","""Advance""",140000
34.4,55.5,"""Chjat""","""Advance""",127100
37.6,55.8,"""Moscou""","""Advance""",100000
34.3,55.2,"""Wixma""","""Retreat""",55000
32.0,54.6,"""Smolensk""","""Retreat""",24000
30.4,54.4,"""Orscha""","""Retreat""",20000
26.8,54.3,"""Moiodexno""","""Retreat""",12000


A common pattern is to assign the original name `minard` to the new DataFrame, so that all future uses of `minard` will refer to the renamed DataFrame.

In [13]:
minard = minard.rename({"City": "City Name"})
minard

Longitude,Latitude,City Name,Direction,Survivors
f64,f64,str,str,i64
32.0,54.8,"""Smolensk""","""Advance""",145000
33.2,54.9,"""Dorogobouge""","""Advance""",140000
34.4,55.5,"""Chjat""","""Advance""",127100
37.6,55.8,"""Moscou""","""Advance""",100000
34.3,55.2,"""Wixma""","""Retreat""",55000
32.0,54.6,"""Smolensk""","""Retreat""",24000
30.4,54.4,"""Orscha""","""Retreat""",20000
26.8,54.3,"""Moiodexno""","""Retreat""",12000


**Accessing the Data in a Column**

We can use a column's label to access the Polars Series of data in the column.

In [14]:
minard.get_column("Survivors")

Survivors
i64
145000
140000
127100
100000
55000
24000
20000
12000


The 5 columns are indexed 0, 1, 2, 3, and 4. The column `Survivors` can also be accessed by using its column index.

In [15]:
minard.to_series(4)

Survivors
i64
145000
140000
127100
100000
55000
24000
20000
12000


The 8 items in the Polars Series are indexed 0, 1, 2, and so on, up to 7. The items in the column can be accessed using `item`.

In [16]:
minard.to_series(4).item(0)

145000

In [17]:
minard.to_series(4).item(5)

24000

**Working with the Data in a Column**

Because columns are Polars Series, we can use Series operations on them to discover new information. For example, we can create a new column that contains the percent of all survivors at each city after Smolensk.

In [18]:
initial = minard.get_column("Survivors").item(0)
minard = minard.with_columns(
    (pl.col("Survivors") / initial).alias("Percent Surviving")
)
minard

Longitude,Latitude,City Name,Direction,Survivors,Percent Surviving
f64,f64,str,str,i64,f64
32.0,54.8,"""Smolensk""","""Advance""",145000,1.0
33.2,54.9,"""Dorogobouge""","""Advance""",140000,0.965517
34.4,55.5,"""Chjat""","""Advance""",127100,0.876552
37.6,55.8,"""Moscou""","""Advance""",100000,0.689655
34.3,55.2,"""Wixma""","""Retreat""",55000,0.37931
32.0,54.6,"""Smolensk""","""Retreat""",24000,0.165517
30.4,54.4,"""Orscha""","""Retreat""",20000,0.137931
26.8,54.3,"""Moiodexno""","""Retreat""",12000,0.082759


[POLARS_CHANGE] To make the proportions in the new column appear as percents, multiply by 100, round to two decimal places, and append a percent sign. `with_columns` returns a new DataFrame, so the result is assigned back to `minard`.

In [19]:
minard = minard.with_columns(
    ((pl.col("Percent Surviving") * 100).round(2).cast(pl.String) + "%").alias("Percent Surviving")
)
minard

Longitude,Latitude,City Name,Direction,Survivors,Percent Surviving
f64,f64,str,str,i64,str
32.0,54.8,"""Smolensk""","""Advance""",145000,"""100.0%"""
33.2,54.9,"""Dorogobouge""","""Advance""",140000,"""96.55%"""
34.4,55.5,"""Chjat""","""Advance""",127100,"""87.66%"""
37.6,55.8,"""Moscou""","""Advance""",100000,"""68.97%"""
34.3,55.2,"""Wixma""","""Retreat""",55000,"""37.93%"""
32.0,54.6,"""Smolensk""","""Retreat""",24000,"""16.55%"""
30.4,54.4,"""Orscha""","""Retreat""",20000,"""13.79%"""
26.8,54.3,"""Moiodexno""","""Retreat""",12000,"""8.28%"""


**Choosing Sets of Columns**

The method `select` creates a new DataFrame that contains only the specified columns.

In [20]:
minard.select("Longitude", "Latitude")

Longitude,Latitude
f64,f64
32.0,54.8
33.2,54.9
34.4,55.5
37.6,55.8
34.3,55.2
32.0,54.6
30.4,54.4
26.8,54.3


The same selection can be made using column indices instead of labels.

In [21]:
minard.select(pl.nth(0), pl.nth(1))

Longitude,Latitude
f64,f64
32.0,54.8
33.2,54.9
34.4,55.5
37.6,55.8
34.3,55.2
32.0,54.6
30.4,54.4
26.8,54.3


The result of using `select` is a new DataFrame, even when you select just one column.

In [22]:
minard.select("Survivors")

Survivors
i64
145000
140000
127100
100000
55000
24000
20000
12000


Notice that the result is a DataFrame, unlike the result of `get_column`, which is a Polars Series.

In [23]:
minard.get_column("Survivors")

Survivors
i64
145000
140000
127100
100000
55000
24000
20000
12000


Another way to create a new DataFrame consisting of a set of columns is to `drop` the columns you don't want.

In [24]:
minard.drop("Longitude", "Latitude", "Direction")

City Name,Survivors,Percent Surviving
str,i64,str
"""Smolensk""",145000,"""100.0%"""
"""Dorogobouge""",140000,"""96.55%"""
"""Chjat""",127100,"""87.66%"""
"""Moscou""",100000,"""68.97%"""
"""Wixma""",55000,"""37.93%"""
"""Smolensk""",24000,"""16.55%"""
"""Orscha""",20000,"""13.79%"""
"""Moiodexno""",12000,"""8.28%"""


Neither `select` nor `drop` change the original DataFrame. Instead, they create new smaller DataFrames that share the same data. The fact that the original DataFrame is preserved is useful! You can generate multiple different DataFrames that only consider certain columns without worrying that one analysis will affect the other.

In [25]:
minard

Longitude,Latitude,City Name,Direction,Survivors,Percent Surviving
f64,f64,str,str,i64,str
32.0,54.8,"""Smolensk""","""Advance""",145000,"""100.0%"""
33.2,54.9,"""Dorogobouge""","""Advance""",140000,"""96.55%"""
34.4,55.5,"""Chjat""","""Advance""",127100,"""87.66%"""
37.6,55.8,"""Moscou""","""Advance""",100000,"""68.97%"""
34.3,55.2,"""Wixma""","""Retreat""",55000,"""37.93%"""
32.0,54.6,"""Smolensk""","""Retreat""",24000,"""16.55%"""
30.4,54.4,"""Orscha""","""Retreat""",20000,"""13.79%"""
26.8,54.3,"""Moiodexno""","""Retreat""",12000,"""8.28%"""


All of the methods that we have used above can be applied to any DataFrame.